#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [2]:
import json
import sys
from datetime import datetime, date, timedelta
from typing import Dict, List, Any, Optional

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, udf, lit, when, isnan, isnull, to_json, from_json,
    explode, collect_list, struct, array, size, coalesce,
    to_timestamp, datediff, current_date, regexp_extract, broadcast, get_json_object
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, BooleanType, MapType, ArrayType, DoubleType
)
from pyspark.sql import functions as F

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def read_source_data(ds, column, source_table="risk.dws_risk_br_openfinance_request_df"):
    """
    从源表读取数据
    Args:
        source_table: 源表名称
    Returns:
        DataFrame包含所需的列
    """

    df = spark.table(source_table)
    # 选择所需的列
    selected_df = df.select(
        col("flow_no"),
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        col(f"{column}"),
        col("grass_date")
    ).filter(
        col(f"{column}").isNotNull() &
        (col(f"{column}") != "") &
        (col("grass_date") == ds)
    )

    return selected_df


"""
开放金融账户和交易数据解析器

根据开放银行API响应数据创建Account和Transaction类
"""

import json
from typing import List, Optional, Dict, Any
from decimal import Decimal
from datetime import datetime, timezone, timedelta
import pandas as pd
import numpy as np


def get_decimal_amount(amount_info: Dict[str, Any]) -> Optional[Decimal]:
    """从金额信息dict中获取Decimal类型的金额"""
    if not amount_info or amount_info.get("amount") is None:
        return None
    try:
        return Decimal(amount_info["amount"])
    except:
        return None


def create_partie_info(txn_data: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """从交易数据中创建交易对方信息dict"""
    if not any(key.startswith("partie") for key in txn_data.keys()):
        return None

    return {
        "branch_code": txn_data.get("partieBranchCode"),
        "check_digit": txn_data.get("partieCheckDigit"),
        "cnpj_cpf": txn_data.get("partieCnpjCpf"),
        "compe_code": txn_data.get("partieCompeCode"),
        "number": txn_data.get("partieNumber"),
        "person_type": txn_data.get("partiePersonType")
    }


def create_transaction(txn_data: Dict[str, Any], account_id: str,
                       institution_name: str) -> Dict[str, Any]:
    """
    创建交易dict

    整合来自以下API端点的数据：
    - accounts/v2/accounts/{accountId}/transactions
    - accounts/v2/accounts/{accountId}/transactions-current
    """
    # 处理transaction_id
    transaction_id = txn_data.get("transactionId")
    if isinstance(transaction_id, list) and transaction_id:
        transaction_id = transaction_id[0].get("value", "")

    # 处理交易对方信息
    partie_info = create_partie_info(txn_data)

    # 处理交易金额信息
    transaction_amount = txn_data.get("transactionAmount", {})
    if isinstance(transaction_amount, str):
        try:
            transaction_amount = json.loads(transaction_amount)
        except:
            transaction_amount = {}

    return {
        # 基本交易信息
        "transaction_id": transaction_id,
        "completed_authorised_payment_type": txn_data.get("completedAuthorisedPaymentType"),
        "credit_debit_type": txn_data.get("creditDebitType"),
        "transaction_name": txn_data.get("transactionName"),
        "type": txn_data.get("type"),
        "transaction_amount": transaction_amount,
        "transaction_date_time": txn_data.get("transactionDateTime"),

        # 交易对方信息
        "partie_info": partie_info,

        # 元数据
        "account_id": account_id,
        "institution_name": institution_name,
    }


def get_transaction_amount_decimal(transaction: Dict[str, Any]) -> Optional[Decimal]:
    """获取交易金额的Decimal值"""
    return get_decimal_amount(transaction.get("transaction_amount", {}))


def get_transaction_display_name(transaction: Dict[str, Any]) -> str:
    """获取交易显示名称"""
    direction = "收入" if is_credit(transaction) else "支出"
    amount = get_transaction_amount_decimal(transaction)
    transaction_amount = transaction.get("transaction_amount", {})
    currency = transaction_amount.get("currency", "") if transaction_amount else ""
    amount_str = f"{amount} {currency}" if amount else "金额未知"
    return f"{direction}: {transaction.get('transaction_name') or transaction.get('type')} - {amount_str}"


def is_credit(transaction: Dict[str, Any]) -> bool:
    """判断是否为收入交易"""
    credit_debit_type = transaction.get("credit_debit_type", "")
    return credit_debit_type.lower() == "credito"


def extract_account_id_from_url(url: str, suffix: str) -> str:
    """从URL中提取account ID"""
    parts = url.split("/accounts/v2/accounts")
    if len(parts) > 1:
        account_part = parts[1].replace(f"{suffix}", "")
        account_id = account_part.strip("/")
        return account_id
    return ""


def parse_transactions_from_response(rsp: List[Dict], account_id: str, institution_name: str) -> \
List[Dict[str, Any]]:
    """解析交易记录响应"""
    transactions = []
    if not isinstance(rsp, list):
        return transactions

    for txn in rsp:
        transaction = create_transaction(txn, account_id, institution_name)
        transactions.append(transaction)

    return transactions


def parse_institution_data(institution_name: str, institution_data: List[Dict]) -> List[Dict[str, Any]]:
    """解析单个机构的数据"""
    transactions = []
    current_transactions = []
    for api_call in institution_data:
        if isinstance(api_call, str):
            api_call = json.loads(api_call)
        url = api_call.get("url", "")
        rsp = api_call.get("rsp")
        if isinstance(rsp, str):
            rsp = json.loads(rsp)
        rsp_data = rsp.get("data", {})
        request_time = api_call.get("request_time", "")

        if not rsp:
            continue

        # 解析不同类型的API响应 -- transactions-current 最近7天内，trasnactions 最近12个月，两个是包含于关系
        if "/transactions-current" in url:
            # 交易记录API
            account_id = extract_account_id_from_url(url, suffix="transactions-current")
            institution_transactions = parse_transactions_from_response(rsp_data, account_id, institution_name)
            current_transactions.extend(institution_transactions)
        elif "/transactions" in url:
            account_id = extract_account_id_from_url(url, suffix="transactions")
            institution_transactions = parse_transactions_from_response(rsp_data, account_id, institution_name)
            transactions.extend(institution_transactions)

    return transactions, current_transactions


def parse_open_finance_data(json_obj: dict) -> List[Dict[str, Any]]:
    """开放金融数据解析器主函数"""
    transactions = []
    current_transactions = []
    data = json_obj.get("get_open_finance_personal_account_data", [])
    for institution in data:
        # logger.info(f"institution: {institution}")
        if isinstance(institution, str):
            institution = json.loads(institution)
        organisation_name = institution.get("organisation_name", "")

        institution_transactions, current_transactions = parse_institution_data(organisation_name, institution.get("data", []))
        transactions.extend(institution_transactions)

    # logger.info(f"transactions: {transactions}")
    return transactions,current_transactions


time_windows = [7, 30, 90, 180, 360]

all_inflow_ft = []

type_list = [
    "TED"
    , "DOC"
    , "PIX"
    , "TRANSFERENCIA_MESMA_INSTITUICAO"
    , "BOLETO"
    , "CONVENIO_ARRECADACAO"
    , "PACOTE_TARIFA_SERVICOS"
    , "TARIFA_SERVICOS_AVULSOS"
    , "FOLHA_PAGAMENTO"
    , "DEPOSITO"
    , "SAQUE"
    , "CARTAO"
    , "ENCARGOS_JUROS_CHEQUE_ESPECIAL"
    , "RENDIMENTO_APLIC_FINANCEIRA"
    , "PORTABILIDADE_SALARIO"
    , "RESGATE_APLIC_FINANCEIRA"
    , "OPERACAO_CREDITO"
    , "OUTROS"
]
party_type_list = [
    "PESSOA_NATURAL"
    , "PESSOA_JURIDICA"
]


def update_inflow_null_ft():
    null_dict = {}
    prefix = "inst_txn_inflow"
    null_dict.update(update_days_from_360d_null_ft("first", prefix))
    null_dict.update(update_days_from_360d_null_ft("last", prefix))
    for interval in time_windows:
        null_dict.update(update_amt_null_ft(interval, "", prefix))
        null_dict.update(update_cnt_null_ft(interval, "", prefix))
        for type in type_list:
            null_dict.update(update_amt_null_ft(interval, type.lower(), prefix))
            null_dict.update(update_cnt_null_ft(interval, type.lower(), prefix))
            null_dict.update(update_ratio_null_ft(interval, type.lower(), "amt", prefix))
            null_dict.update(update_ratio_null_ft(interval, type.lower(), "cnt", prefix))


        for party in party_type_list:
            null_dict.update(update_amt_null_ft(interval, party.lower(), prefix))
            null_dict.update(update_cnt_null_ft(interval, party.lower(), prefix))
            null_dict.update(update_ratio_null_ft(interval, party.lower(), "amt", prefix))
            null_dict.update(update_ratio_null_ft(interval, party.lower(), "cnt", prefix))

    return null_dict


def update_days_from_360d_null_ft(adj, prefix):
    return {
        f"{prefix}_days_from_{adj}_txn_360d": None
    }


def update_amt_null_ft(interval, cond_prefix="", prefix="inst_txn_inflow"):
    return {
        f"{prefix}_{cond_prefix + '_' if cond_prefix else ''}amt_total_{interval}d": None,
        f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}amt_avg_{interval}d': None,
        f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}amt_max_{interval}d': None,
        f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}amt_min_{interval}d': None
    }


def update_cnt_null_ft(interval, cond_prefix="", prefix="inst_txn_inflow"):
    return {
        f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}cnt_{interval}d': None
    }


def update_ratio_null_ft(interval, cond_prefix="", cat="", prefix="inst_txn_inflow"):
    return {
        f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}{cat}_ratio_{interval}d': None
    }


def get_days_from_360d(df: pd.DataFrame, adj, prefix):
    if df is None:
        return {
            f"{prefix}_days_from_{adj}_txn_360d": None
        }
    df_filtered = df[df["update_diff_days"] <= 360]
    if adj == "first":
        days = df_filtered["update_diff_days"].max()
    elif adj == "last":
        days = df_filtered["update_diff_days"].min()
    return {
        f"{prefix}_days_from_{adj}_txn_360d": None if np.isnan(days) else int(days)
    }


def get_amt_profile(df: pd.DataFrame, interval, cond_prefix="", cat="", prefix=""):
    if df is None:
        return {
            f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}amt_total_{interval}d': None,
            f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}amt_avg_{interval}d': None,
            f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}amt_max_{interval}d': None,
            f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}amt_min_{interval}d': None
        }
    df_filter = df[df["update_diff_days"] <= interval]

    if cond_prefix == "":
        sum = df_filter["amount"].sum()
        max = df_filter["amount"].max()
        min = df_filter["amount"].min()
        avg = df_filter["amount"].sum() / df_filter["amount"].count() if len(df_filter["amount"]) > 0 else 0
        return {
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_total_{interval}d': float(sum),
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_avg_{interval}d': float(avg),
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_max_{interval}d': float(
                max) if max is not np.nan else 0,
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_min_{interval}d': float(
                min) if max is not np.nan else 0
        }
    else:
        df_cond = df_filter[df_filter[cat] == cond_prefix]
        shape = df_cond.shape
        if shape[0] == 0:
            return {
                f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_total_{interval}d': float(0),
                f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_avg_{interval}d': float(0),
                f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_max_{interval}d': float(0),
                f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_min_{interval}d': float(0)
            }
        sum = df_cond["amount"].sum()
        max = df_cond["amount"].max()
        min = df_cond["amount"].min()
        avg = df_cond["amount"].sum() / df_cond["amount"].count() if len(df_cond["amount"]) > 0 else 0
        return {
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_total_{interval}d': float(sum),
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_avg_{interval}d': float(avg),
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_max_{interval}d': float(max),
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}amt_min_{interval}d': float(min),

        }


def get_cnt_profile(df: pd.DataFrame, interval, cond_prefix="", cat="", prefix="inst_txn_inflow"):
    if df is None:
        return {
            f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}cnt_{interval}d': None
        }

    df_filter = df[df["update_diff_days"] <= interval]
    if cond_prefix == "":
        cnt = df_filter.count().values[0]
        return {
            f'{prefix}_{cond_prefix + "_" if cond_prefix else ""}cnt_{interval}d': int(cnt)
        }
    else:
        df_cond = df_filter[df_filter[cat] == cond_prefix]
        shape = df_cond.shape
        if shape[0] == 0:
            return {
                f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}cnt_{interval}d': 0
            }
        cnt = shape[0]
        return {
            f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}cnt_{interval}d': int(cnt)
        }


def get_ratio_profile(all: dict, type: dict, interval, cond_prefix, cat, prefix):
    type_value = 0
    all_value = 0
    if cat == "cnt":
        type_value = type[f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}{cat}_{interval}d']
        all_value = all.get(f'{prefix}_{cat}_{interval}d', 0)
    elif cat == "amt":
        type_value = type[f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}{cat}_total_{interval}d']
        all_value = all.get(f'{prefix}_{cat}_total_{interval}d', 0)

    ratio = type_value / all_value if all_value != 0 else 0
    return {
        f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}{cat}_ratio_{interval}d': float(ratio)
    }


def safe_to_float(series):
    return series.replace('', np.nan).astype(float)


def safe_to_date(date):
    if date is None:
        return None
    try:
        utc_time = datetime.fromisoformat(date.replace("Z", "+00:00"))

        # 巴西时间 (UTC-3)
        brt = timezone(timedelta(hours=-3))
        brt_time = utc_time.astimezone(brt).replace(hour=0, minute=0, second=0, microsecond=0)

        return brt_time.strftime("%Y-%m-%d")
    except Exception as e:
        print(f"日期转换异常: {date}, 错误: {e}")
        return None


def get_cpf_inflow_transaction_nday_profile(trans):
    pd_trans = pd.DataFrame(trans)

    if pd_trans.empty:
        return update_inflow_null_ft()

    pd_trans["amount"] = safe_to_float(pd_trans["transaction_amount"])
    pd_trans["update_diff_days"] = (pd.to_datetime(pd_trans['request_date']) -
                                    pd.to_datetime(pd_trans['transaction_date'],
                                                   errors='coerce')).dt.days
    prefix = "inst_txn_inflow"
    result = {}
    for interval in time_windows:
        # days from
        result.update(get_days_from_360d(pd_trans, "first", prefix))
        result.update(get_days_from_360d(pd_trans, "last", prefix))
        # cnt
        cnt = get_cnt_profile(pd_trans, interval, "", "",prefix=prefix)
        result.update(cnt)
        # amt
        amt = get_amt_profile(pd_trans, interval, "", "",prefix=prefix)
        result.update(amt)
        # type
        for type in type_list:
            type_amt = get_amt_profile(pd_trans, interval, type.lower(), "type", prefix)
            type_cnt = get_cnt_profile(pd_trans, interval, type.lower(), "type", prefix)
            amt_ratio = get_ratio_profile(amt, type_amt, interval, type.lower(), "amt", prefix)
            cnt_ratio = get_ratio_profile(cnt, type_cnt, interval, type.lower(), "cnt", prefix)
            result.update(type_amt)
            result.update(type_cnt)
            result.update(amt_ratio)
            result.update(cnt_ratio)

        for party_type in party_type_list:
            type_amt = get_amt_profile(pd_trans, interval, party_type.lower(), "party_type", prefix)
            type_cnt = get_cnt_profile(pd_trans, interval, party_type.lower(), "party_type", prefix)
            amt_ratio = get_ratio_profile(amt, type_amt, interval, party_type.lower(), "amt", prefix)
            cnt_ratio = get_ratio_profile(cnt, type_cnt, interval, party_type.lower(), "cnt", prefix)
            result.update(type_amt)
            result.update(type_cnt)
            result.update(amt_ratio)
            result.update(cnt_ratio)

    return result


def update_outflow_null_ft():
    null_dict = {}
    prefix = "inst_txn_outflow"
    null_dict.update(update_days_from_360d_null_ft("first", prefix))
    null_dict.update(update_days_from_360d_null_ft("last", prefix))
    for interval in time_windows:
        null_dict.update(update_amt_null_ft(interval, "", prefix))
        null_dict.update(update_cnt_null_ft(interval, "", prefix))
        for type in type_list:
            null_dict.update(update_amt_null_ft(interval, type.lower(), prefix))
            null_dict.update(update_cnt_null_ft(interval, type.lower(), prefix))
            null_dict.update(update_ratio_null_ft(interval, type.lower(), "amt",prefix))
            null_dict.update(update_ratio_null_ft(interval, type.lower(), "cnt",prefix))


        for party in party_type_list:
            null_dict.update(update_amt_null_ft(interval, party.lower(), prefix))
            null_dict.update(update_cnt_null_ft(interval, party.lower(), prefix))
            null_dict.update(update_ratio_null_ft(interval, type.lower(), "amt",prefix))
            null_dict.update(update_ratio_null_ft(interval, type.lower(), "cnt",prefix))

    return null_dict


def get_cpf_outflow_transaction_nday_profile(trans):
    pd_trans = pd.DataFrame(trans)

    if pd_trans.empty:
        return update_outflow_null_ft()

    pd_trans["amount"] = safe_to_float(pd_trans["transaction_amount"])
    try:
        pd_trans["update_diff_days"] = (pd.to_datetime(pd_trans['request_date']) -
                                    pd.to_datetime(pd_trans['transaction_date'],
                                                   errors='coerce')).dt.days
    except TypeError as e:
        return f"{trans}"
    prefix = "inst_txn_outflow"
    result = {}
    for interval in time_windows:
        # days from
        result.update(get_days_from_360d(pd_trans, "first", prefix))
        result.update(get_days_from_360d(pd_trans, "last", prefix))
        # cnt
        cnt = get_cnt_profile(pd_trans, interval, "","", prefix)
        result.update(cnt)
        # amt
        amt = get_amt_profile(pd_trans, interval, "", "",prefix)
        logger.info(f"{amt}")
        result.update(amt)
        # type
        for type in type_list:
            type_amt = get_amt_profile(pd_trans, interval, type.lower(), "type", prefix)
            type_cnt = get_cnt_profile(pd_trans, interval, type.lower(), "type", prefix)
            amt_ratio = get_ratio_profile(amt, type_amt, interval, type, "amt", prefix)
            cnt_ratio = get_ratio_profile(cnt, type_cnt, interval, type, "cnt", prefix)
            result.update(type_amt)
            result.update(type_cnt)
            result.update(amt_ratio)
            result.update(cnt_ratio)

        for party_type in party_type_list:
            type_amt = get_amt_profile(pd_trans, interval, party_type.lower(), "party_type", prefix)
            type_cnt = get_cnt_profile(pd_trans, interval, party_type.lower(), "party_type", prefix)
            amt_ratio = get_ratio_profile(amt, type_amt, interval, party_type, "amt", prefix)
            cnt_ratio = get_ratio_profile(cnt, type_cnt, interval, party_type, "cnt", prefix)
            result.update(type_amt)
            result.update(type_cnt)
            result.update(amt_ratio)
            result.update(cnt_ratio)
    return result


def get_netflow_amt_ft(inflow_fts: dict, outflow_fts: dict, interval, cond_prefix="",
                       prefix="inst_txn_netflow") -> dict:
    inflows_ft = inflow_fts.get(f'inst_txn_inflow_{cond_prefix + "_" if cond_prefix else ""}amt_total_{interval}d', 0)
    outflows_ft = outflow_fts.get(f'inst_txn_outflow_{cond_prefix + "_" if cond_prefix else ""}amt_total_{interval}d',
                                  0)
    if inflows_ft is None and outflows_ft is None:
        netflow_ft = None
    elif inflows_ft is None:
        netflow_ft = 0 - outflows_ft
    elif outflows_ft is None:
        netflow_ft = inflows_ft
    else:
        netflow_ft = float(inflows_ft - outflows_ft)

    return {
        f'{prefix}_{cond_prefix.lower() + "_" if cond_prefix.lower() else ""}total_amt_{interval}d': netflow_ft,
    }


def get_cpf_netflow_transaction_nday_profile(inflow_fts, outflow_fts):
    prefix = "inst_txn_netflow"
    result = {}
    for interval in time_windows:

        amt = get_netflow_amt_ft(inflow_fts, outflow_fts, interval, "", "inst_txn_netflow")
        result.update(amt)
        for type in type_list:
            type_amt = get_netflow_amt_ft(inflow_fts, outflow_fts, interval, type.lower(), "inst_txn_netflow")
            result.update(type_amt)
        for party_type in party_type_list:
            party_type_amt = get_netflow_amt_ft(inflow_fts, outflow_fts, interval, party_type.lower(),
                                                "inst_txn_netflow")
            result.update(party_type_amt)

    return result


def get_distinct_transaction_list(transactions, current_transactions):
    txn_ids = []
    distinct_txns = []
    for txn in current_transactions:
        txn_id = txn["transaction_id"]
        if txn_id not in txn_ids:
            distinct_txns.append(txn)
            txn_ids.append(txn_id)

    for txn in transactions:
        txn_id = txn["transaction_id"]
        if txn_id not in txn_ids:
            distinct_txns.append(txn)
            txn_ids.append(txn_id)

    return distinct_txns
            


def main(data, cpf, request_date):
    """主函数 - 演示如何使用解析器"""

    # 解析数据
    data = json.loads(data)
    transactions,current_transactions = parse_open_finance_data(data)
    transaction_list = get_distinct_transaction_list(transactions, current_transactions)
    inflow = []
    outflow = []
    trans = []
    for transaction in transaction_list:
        credit_debit_type = transaction["credit_debit_type"]
        if credit_debit_type.lower() == "CREDITO".lower():
            inflow.append(transaction)
        elif credit_debit_type.lower() == "DEBITO".lower():
            outflow.append(transaction)

    outflow_result = update_outflow_null_ft()
    result_all = {}

    if len(outflow) == 0:
        outflow_result = update_outflow_null_ft()
    else:
        trans = []
        for transaction in outflow:
            transaction_id = transaction["transaction_id"]
            transaction_type = transaction["type"].lower()
            transaction_date_time = safe_to_date(transaction.get("transaction_date_time", pd.NaT))
            party_info = transaction.get("partie_info", {})
            if party_info is None:
                party_type = None
            else:
                party_type = party_info.get("person_type", None)
            if isinstance(transaction["transaction_amount"], str):
                amt = json.loads(transaction["transaction_amount"])
            else:
                amt = transaction["transaction_amount"]
            amt_value = amt.get("amount", 0)
            amt_currency = amt.get("currency", None)
            t = {
                "transaction_id": transaction_id,
                "transaction_date": transaction_date_time,
                "party_type": party_type.lower() if isinstance(party_type, str) else None,
                "type": transaction_type,
                "transaction_amount": amt_value,
                "currency": amt_currency,
                "request_date": request_date
            }
            trans.append(t)
        outflow_result = get_cpf_outflow_transaction_nday_profile(trans)
    result_all.update(outflow_result)

    if len(inflow) == 0:
        inflow_result = update_inflow_null_ft()
    else:
        trans = []
        for transaction in inflow:
            transaction_id = transaction["transaction_id"]
            transaction_type = transaction["type"].lower()
            transaction_date_time = safe_to_date(transaction.get("transaction_date_time", pd.NaT))
            party_info = transaction.get("partie_info", {})
            if party_info is None:
                party_type = None
            else:
                party_type = party_info.get("person_type", None)
            if isinstance(transaction["transaction_amount"], str):
                amt = json.loads(transaction["transaction_amount"])
            else:
                amt = transaction["transaction_amount"]
            amt_value = amt["amount"]
            amt_currency = amt["currency"]
            t = {
                "transaction_id": transaction_id,
                "transaction_date": transaction_date_time,
                "party_type": party_type.lower() if isinstance(party_type, str) else None,
                "type": transaction_type,
                "transaction_amount": amt_value,
                "currency": amt_currency,
                "request_date": request_date
            }
            trans.append(t)
        inflow_result = get_cpf_inflow_transaction_nday_profile(trans)

    result_all.update(inflow_result)

    # 计算netflow特征（现在变量已确保定义）
    net_result = get_cpf_netflow_transaction_nday_profile(inflow_result, outflow_result)
    result_all.update(net_result)
    
    return {
        "transaction_list": result_all,
    }

def get_all_null_ft():
    result_all = {}
    outflow_result = update_outflow_null_ft()
    result_all.update(outflow_result)

    inflow_result = update_inflow_null_ft()
    result_all.update(inflow_result)

    net_result = get_cpf_netflow_transaction_nday_profile(inflow_result, outflow_result)

    result_all.update(net_result)

    return result_all


def process_row(raw_data, cpf, request_date):
        
    result = main(raw_data, cpf, request_date)
    # 确保返回值格式正确
    feature_group = result.get("transaction_list", get_all_null_ft)
    # 将所有值转换为字符串以符合MapType定义
    feature_group = {k: str(v) if v is not None else None for k, v in feature_group.items()}

    return {
        "feature_group": feature_group
    }

def process_dataframe(input_df):
    """
    处理输入DataFrame

    Args:
        input_df: 输入DataFrame，包含shopee_user_id, cpf, get_open_finance_personal_account_data

    Returns:
        处理后的DataFrame，包含shopee_user_id, cpf, origin_data, feature_group
    """
    logger.info("开始处理DataFrame...")

    # 检查输入数据
    input_count = input_df.count()
    logger.info(f"输入数据记录数: {input_count}")

    # 缓存输入DataFrame以提高性能
    input_df.cache()

    # 定义返回类型
    return_schema = StructType([
        StructField("feature_group", MapType(StringType(), StringType(), True), True)
    ])
    # 创建处理UDF
    process_udf = udf(process_row, return_schema)


    # 应用UDF处理数据，添加重分区以优化性能
    result_df = input_df.repartition(200).withColumn(
        "processed_result",
        process_udf(
            col("get_open_finance_personal_account_data"),
            col("cpf"),
            col("request_date")
        )
    )

    # 展开处理结果
    final_df = result_df.select(
        col("flow_no"),
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        to_json(col("processed_result.feature_group")).alias("feature_group"),
        col("get_open_finance_personal_account_data").alias("origin_data")

    )

    # 缓存结果DataFrame
    final_df.cache()

    # 检查输出数据质量
    output_count = final_df.count()
    logger.info(f"输出数据记录数: {output_count}")
    
    return final_df

In [3]:
df = read_source_data(ds='${ds}', column="get_open_finance_personal_account_data")
final_df = process_dataframe(df)

In [4]:

def write_to_hive(spark_df, category='', schema="dev_risk", table="ads_risk_br_openfinance_request_df", env="",
                  ds="2025-09-17"):
    """
    将 Spark DataFrame 写入 Hive 表 (支持分区覆盖)
    """

    root_path = f"/projects/risk/hdfs/{env}/risk_mart/ads/ads_risk_openfinance_request_df/grass_region=BR"
    # 添加分区字段 grass_date
    spark_df_with_partition = spark_df \
        .withColumn("grass_date", F.lit(ds)) \
        .withColumn("category", F.lit(category))

    # 覆盖写入 Hive (支持分区)
    (
        spark_df_with_partition
        .write.mode("overwrite")
        .format("parquet")
        .partitionBy("category", "grass_date")
        .option("partitionOverwriteMode", "dynamic")
        .option("parquet.compression", "snappy")  # 避免 zstd 兼容问题
        .option("spark.sql.parquet.writeLegacyFormat", "true")  # 避免 decimal/时间戳兼容
        .parquet(f"{root_path}")
    )

    print(f"✅ Data written to Hive table: {schema}.{table} partition grass_date={ds}")
    print(f"📊 Total records written: {spark_df_with_partition.count()}")

    # 如果 Hive Metastore 需要同步 HDFS 分区，可以在 Spark SQL 里调用 MSCK
    spark.sql(f"MSCK REPAIR TABLE {schema}.{table}")

In [5]:
write_to_hive(final_df, category='transaction_profile', schema='${SCHEMA}', table="ads_risk_br_openfinance_request_df",
                  env='${ENV}', ds='${ds}')

✅ Data written to Hive table: dev_risk.ads_risk_br_openfinance_request_df partition grass_date=2025-11-12
📊 Total records written: 1000